# Notebook 5 — Modern Pretrained Backbones: ResNet-50, EfficientNetV2 & ConvNeXt

## Story
Lightweight models (MobileNet, EfficientNet-B0) showed the power of transfer
learning.  Now we push further with **larger, more powerful backbones** and
a more sophisticated fine-tuning strategy:

| Model | Params | Highlight |
|---|---|---|
| **ResNet-50** | ~25 M | Deep residual network; widely used baseline. |
| **EfficientNetV2-S** | ~22 M | Faster training via progressive learning & Fused-MBConv. |
| **ConvNeXt-Tiny** | ~28 M | Pure-CNN re-designed to match ViT: LayerNorm, GELU, large kernels. |

**Two-stage fine-tuning** strategy:
1. **Warmup (frozen backbone):** train only the new head for a few epochs.
   Prevents random-init gradients from destroying ImageNet features.
2. **Full fine-tune:** unfreeze the entire backbone with a 10× smaller LR
   than the head (discriminative LRs).

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from torchvision import models as tv_models
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from torch.utils.data import DataLoader

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, subsample_subset,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines, print_model_info,
    save_checkpoint, load_checkpoint,
    plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, show_saliency_examples,
    compute_multilabel_metrics, NUM_LABELS, METRIC_KEYS,
    NORM_MEAN, NORM_STD, TransformSubset,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 224      # native ImageNet resolution
BATCH_SIZE      = 32       # smaller batch due to larger images
SPLIT           = [0.7, 0.15, 0.15]
USE_SUBSET      = False
SUBSET_FRACTION = 0.1
CHECKPOINT_DIR  = Path("../checkpoints")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 3. Data Loading — Stratified Split + RandAugment

We use a **stratified split** so that every rare label combination is
represented in all three partitions.  We also switch from hand-crafted
augmentations to **RandAugment** + **RandomErasing**.

In [ ]:
from collections import defaultdict
import numpy as np
from torch.utils.data import Subset


def stratified_split_multilabel(dataset, split, seed=42):
    """Stratified split that keeps every label combo in all partitions."""
    combo_to_indices = defaultdict(list)
    for i in range(len(dataset)):
        _, target = dataset[i]
        combo_to_indices[tuple(target.int().tolist())].append(i)

    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for indices in combo_to_indices.values():
        indices = np.array(indices)
        rng.shuffle(indices)
        n       = len(indices)
        n_val   = max(1 if n >= 3 else 0, round(split[1] * n))
        n_test  = max(1 if n >= 3 else 0, round(split[2] * n))
        n_train = max(0, n - n_val - n_test)
        train_idx.extend(indices[:n_train].tolist())
        val_idx.extend(indices[n_train : n_train + n_val].tolist())
        test_idx.extend(indices[n_train + n_val :].tolist())
    return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    RandomErasing(p=0.25, scale=(0.02, 0.1)),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = stratified_split_multilabel(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_ds = TransformSubset(train_raw, transform=train_transform)
val_ds   = TransformSubset(val_raw,   transform=eval_transform)
test_ds  = TransformSubset(test_raw,  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 4. Model Definitions — TransferModel Wrapper

Each backbone is wrapped so we can toggle backbone gradients for the
two-stage fine-tuning procedure.

In [ ]:
class TransferModel(nn.Module):
    """Pretrained backbone + custom head; supports set_backbone_trainable()."""

    def __init__(self, backbone: nn.Module, head: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.head     = head

    def forward(self, x):
        return self.head(self.backbone(x))

    def set_backbone_trainable(self, flag: bool):
        for p in self.backbone.parameters():
            p.requires_grad = flag


def build_model(arch: str, num_labels: int) -> TransferModel:
    if arch == "resnet50":
        m        = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
        backbone = nn.Sequential(*list(m.children())[:-1], nn.Flatten())
        head     = nn.Sequential(nn.Dropout(0.3), nn.Linear(2048, num_labels))

    elif arch == "efficientnet_v2_s":
        m        = tv_models.efficientnet_v2_s(weights=tv_models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, num_labels))

    elif arch == "convnext_tiny":
        m        = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = nn.Sequential(nn.LayerNorm(768), nn.Dropout(0.2), nn.Linear(768, num_labels))

    else:
        raise ValueError(f"Unknown arch: {arch}")

    model = TransferModel(backbone, head)
    # Init head
    for layer in model.head.modules():
        if isinstance(layer, nn.Linear):
            nn.init.normal_(layer.weight, 0, 0.01)
            nn.init.zeros_(layer.bias)
    return model


ARCHS = ["resnet50", "efficientnet_v2_s", "convnext_tiny"]

print(f"{'Arch':<22} {'Params':>12}  Size")
print("-" * 45)
for arch in ARCHS:
    m     = build_model(arch, NUM_LABELS)
    total = sum(p.numel() for p in m.parameters())
    size_mb = total * 4 / 1024 / 1024
    print(f"{arch:<22} {total:>12,}  {size_mb:.1f} MB")

## 5. Two-Stage Training Loop

Stage 1 (freeze): only head trains.  
Stage 2 (unfreeze): backbone trains at `LR_BACKBONE` (10× lower than head).

In [ ]:
FREEZE_EPOCHS   = 3
UNFREEZE_EPOCHS = 15
LR_HEAD         = 1e-3
LR_BACKBONE     = 1e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
THRESHOLD       = 0.5
criterion       = nn.BCEWithLogitsLoss()


def run_epoch(model, loader, optimizer=None, train=True):
    model.train(train)
    all_logits, all_preds, all_labels = [], [], []
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP
                )
                optimizer.step()
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= THRESHOLD).float()
        total_loss += loss.item() * images.size(0)
        all_logits.append(logits.detach().cpu())
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
    n = sum(len(b) for b in all_labels)
    met = compute_multilabel_metrics(
        torch.cat(all_labels), torch.cat(all_preds), logits=torch.cat(all_logits)
    )
    return met


def two_stage_train(arch):
    model = build_model(arch, NUM_LABELS).to(DEVICE)
    model.set_backbone_trainable(False)    # Stage 1: frozen

    history = {k: {"train": [], "val": []} for k in METRIC_KEYS}
    best_f1, best_state = -1.0, None

    for epoch in range(1, FREEZE_EPOCHS + UNFREEZE_EPOCHS + 1):
        # Switch to full fine-tune after warmup
        if epoch == FREEZE_EPOCHS + 1:
            model.set_backbone_trainable(True)
            print(f"  [Epoch {epoch}] Backbone unfrozen — full fine-tune starts.")

        if epoch <= FREEZE_EPOCHS:
            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=LR_HEAD, weight_decay=WEIGHT_DECAY,
            )
        else:
            optimizer = optim.Adam([
                {"params": model.head.parameters(),     "lr": LR_HEAD},
                {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
            ], weight_decay=WEIGHT_DECAY)

        tr  = run_epoch(model, train_loader, optimizer, train=True)
        val = run_epoch(model, val_loader,   optimizer=None, train=False)

        for k in METRIC_KEYS:
            history[k]["train"].append(tr[k])
            history[k]["val"].append(val[k])

        if val["f1_micro"] > best_f1:
            best_f1    = val["f1_micro"]
            best_state = copy.deepcopy(model.state_dict())

        print(f"  Epoch {epoch:>2}  train_f1={tr['f1_micro']:.4f}  val_f1={val['f1_micro']:.4f}")

    return best_state, best_f1, history, FREEZE_EPOCHS + UNFREEZE_EPOCHS

In [ ]:
all_histories = {}
best_val_f1s  = {}
checkpoints   = {}

for arch in ARCHS:
    print(f"\n{'='*55}\n  Training: {arch}\n{'='*55}")
    ckpt = CHECKPOINT_DIR / f"tl_v3_{arch}.pth"
    checkpoints[arch] = ckpt

    best_state, best_f1, history, total_epochs = two_stage_train(arch)
    save_checkpoint(best_state, ckpt)

    all_histories[arch] = history
    best_val_f1s[arch]  = best_f1
    print(f"\n  Best val F1: {best_f1:.4f}  |  checkpoint: {ckpt}")

print("\n=== Val F1 Summary ===")
for arch in ARCHS:
    print(f"  {arch:<22}  {best_val_f1s[arch]:.4f}")

## 6. Training Curves

In [ ]:
plot_multi_arch_histories(all_histories, experiment_name="Modern Backbones (two-stage)")

## 7. Test-Set Metrics

In [ ]:
print(f"{'Model':<22} {'loss':>7} {'exact':>7} {'hamming':>8} {'IoU':>7} {'prec':>7} {'rec':>7} {'F1':>7}")
print("-" * 85)
for arch in ARCHS:
    fn = lambda nl, a=arch: build_model(a, nl)
    m  = load_checkpoint(fn, NUM_LABELS, checkpoints[arch], DEVICE)
    imgs, lbls, preds, probs = collect_test_predictions(m, test_loader, DEVICE)
    all_logits = []
    m.eval()
    with torch.no_grad():
        for images, _ in test_loader:
            all_logits.append(m(images.to(DEVICE)).cpu())
    logits = torch.cat(all_logits)
    met = compute_multilabel_metrics(lbls, preds, logits=logits)
    print(f"{arch:<22} {met['loss']:>7.4f} {met['exact_match']:>7.4f} "
          f"{met['hamming_acc']:>8.4f} {met['mean_iou']:>7.4f} "
          f"{met['precision_micro']:>7.4f} {met['recall_micro']:>7.4f} "
          f"{met['f1_micro']:>7.4f}")

## 8. Detailed Analysis — Best Modern Backbone

In [ ]:
best_arch = max(best_val_f1s, key=best_val_f1s.get)
print(f"Best: {best_arch}  (val F1 = {best_val_f1s[best_arch]:.4f})")

fn    = lambda nl, a=best_arch: build_model(a, nl)
model = load_checkpoint(fn, NUM_LABELS, checkpoints[best_arch], DEVICE)
images, labels, preds, probs = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels, preds)

show_prediction_examples(correct_idx,   images, labels, preds, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images, labels, preds, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images, labels, preds, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels, preds)
plot_confusion_matrices(labels, preds)

In [ ]:
show_saliency_examples(correct_idx,   images, labels, preds, model, "Fully Correct",   n=4, device=DEVICE)
show_saliency_examples(incorrect_idx, images, labels, preds, model, "Fully Incorrect", n=4, device=DEVICE)

## 9. Conclusions

- Modern large-capacity backbones consistently outperform lightweight ones
  on this task.
- **ConvNeXt** — despite being a pure CNN — benefits from ViT-inspired design
  choices (LayerNorm, GELU activations, large kernels) and achieves strong F1.
- Two-stage fine-tuning (freeze → unfreeze with discriminative LRs) is critical
  to avoid destroying pretrained features during the first training steps.

**Next:** We step outside the CNN paradigm entirely and explore
Vision Transformers — both from scratch and pretrained (Notebook 6).